# tfm3lab — run on Colab (GPU)

Bootstrap only: clone the repo, sync the CUDA environment, run the GPU-hungry scripts.
All the real logic lives in `scripts/*.py` and `src/tfm3lab/` — this notebook has no
experiment code of its own, so a bug fixed once in the repo is fixed everywhere.

Prereqs before running this: accept the license at
https://huggingface.co/google/timesfm-3.0-pytorch, and have a Hugging Face token with
read access ready to paste into `hf auth login` below.

Runtime: make sure Colab is set to a GPU runtime (Runtime -> Change runtime type -> T4 or better).

In [1]:
!pip -q install uv
REPO_URL = "<fill in this repo's URL>"  # e.g. git@github.com:you/timesfm3-talk.git
!git clone -q $REPO_URL timesfm3-talk
%cd timesfm3-talk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 71.0 MB/s eta 0:00:00:00:0100:01
/bin/bash: -c: line 1: unexpected EOF while looking for matching `''
/bin/bash: -c: line 2: syntax error: unexpected end of file
[Errno 2] No such file or directory: 'timesfm3-talk'
/content


In [2]:
!uv sync --extra cuda

error: No `pyproject.toml` found in current directory or any parent directory


In [ ]:
# One-time: accept the gated checkpoint's license at the URL above, then authenticate.
!uv run hf auth login

## Optional: share data/ and results/ with your local machine via Drive

Skip this cell to just use Colab's local (ephemeral) disk instead.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["TFM3LAB_DATA_ROOT"] = "/content/drive/MyDrive/timesfm3-talk-data"

## 1. Fetch data

The full MTG backfill (TCGCSV, ~2.5 years) is the slow part on first run — everything
downloaded is cached, so re-running this cell later only fetches what's missing.

In [ ]:
!uv run scripts/00_probe_tcgcsv.py
!uv run scripts/01_fetch_data.py

## 2-5. Run the experiments

In [ ]:
!uv run scripts/02_exp_mtg.py

In [ ]:
!uv run scripts/03_exp_shock.py

In [ ]:
# Pure re-analysis of 02/03's cached predictions — no GPU needed, but harmless to run here too.
!uv run scripts/04_exp_calibration.py

In [ ]:
!uv run scripts/05_exp_covariates.py

## Bring results home

`results/*.parquet` is everything the local machine needs — figures, slides, and the demo
notebook all read only from there. Download the `results/` folder (or, if you mounted
Drive above, just `git pull`/sync it back into your local checkout) and commit it.

In [ ]:
import shutil
shutil.make_archive("/content/results", "zip", "results")
from google.colab import files
files.download("/content/results.zip")